In [1]:
import pandas as pd

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

In [5]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col]=pd.to_datetime(orders[col])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [10]:
delivered_orders = orders[
    (orders["order_status"] == 'delivered') & (orders["order_purchase_timestamp"] >= "2017-01-01") & 
    (orders["order_purchase_timestamp"] <= "2018-08-01")
].copy()

len(delivered_orders)

89860

In [11]:
#Create the column Year-month
delivered_orders["order_year_month"] = delivered_orders["order_purchase_timestamp"].dt.to_period("M")

delivered_orders["order_year_month"].value_counts().sort_index()

order_year_month
2017-01     750
2017-02    1653
2017-03    2546
2017-04    2303
2017-05    3546
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7289
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6099
2018-07    6159
Freq: M, Name: count, dtype: int64

In [12]:
monthly_items = delivered_orders.merge(
    order_items[["order_id", "price"]],
    on="order_id",
    how="inner"
)

In [14]:
monthly_sumary = monthly_items.groupby("order_year_month").agg(total_orders=("order_id" , "nunique"), total_revenue=("price", "sum"))

monthly_sumary["aov"] = (monthly_sumary["total_revenue"] / monthly_sumary["total_orders"]).round(2) 
monthly_sumary["total_revenue"] = monthly_sumary["total_revenue"].round(2)

monthly_sumary

,total_orders,total_revenue,aov
order_year_month,,,
2017-01,750,111798.36,149.06
2017-02,1653,234223.40,141.70
2017-03,2546,359198.85,141.08
2017-04,2303,340669.68,147.92
2017-05,3546,489338.25,138.00
2017-06,3135,421923.37,134.58
2017-07,3872,481604.52,124.38
2017-08,4193,554699.70,132.29
2017-09,4150,607399.67,146.36
